# 固定数据划分

**负责人：** 4 号　**阶段：** 1 数据准备

生成全组唯一的一套数据划分，保证所有模型的结果可以比较。`folds_v1` 由 2 号在 10/27 按下面的任务清单代为生成，请 4 号复核。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 和 2 号确认清洗数据的列和 `group_id` 的含义
- [x] 用 `StratifiedGroupKFold` 按 `group_id` 分组：先分出约 20% 测试集，再把剩下的分成 5 折，同一组的推文不能跨训练/测试或分到不同折
- [x] 检查每一折的类别比例
- [x] 保存前运行 `check_splits(splits, clean)`

## 产出

- `splits/folds_v1.csv`，列为 `id, group_id, split, fold`（测试集 fold = -1）

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## 1. 读取清洗数据

`group_id` 把完全重复、近似重复和 `text_norm` 相同的推文分到同一组。同一组的推文如果一条在训练集、一条在测试集，模型等于提前看过测试题，所以划分必须以组为单位。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from src.config import CLEAN_VERSION, N_FOLDS, SPLIT_VERSION, SPLITS_DIR, TEST_SIZE
from src.data import check_splits

# 按 id 排序，保证每次运行得到同一个划分
clean = load_clean().sort_values("id").reset_index(drop=True)
group_sizes = clean["group_id"].value_counts()
print(f"clean_v{CLEAN_VERSION}: {len(clean):,} 条推文，{clean['group_id'].nunique():,} 个组；"
      f"{int((group_sizes > 1).sum())} 个组不止一条推文，最大的组 {group_sizes.max()} 条")

## 2. 划分

1. **测试集：** `StratifiedGroupKFold` 把全部推文按组分成 1 / `TEST_SIZE` = 5 份，取第 1 份作为测试集（约 20%）。测试集只在 11/18 实验冻结后用一次。
2. **5 折：** 剩下的开发集再用 `StratifiedGroupKFold` 分成 `N_FOLDS` = 5 折，调参、选模型和 Stacking 都只用这 5 折的 OOF 预测。

两步都按组切分，同时让每份的类别比例接近总体（分层），随机种子固定为 `SEED` = 42。

In [ ]:
n_outer = round(1 / TEST_SIZE)
outer = StratifiedGroupKFold(n_splits=n_outer, shuffle=True, random_state=SEED)
dev_idx, test_idx = next(outer.split(clean, clean["label"], clean["group_id"]))

splits = clean[["id", "group_id"]].copy()
splits["split"] = "train"
splits["fold"] = -1
splits.loc[test_idx, "split"] = "test"

dev = clean.iloc[dev_idx]
inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, valid_idx) in enumerate(inner.split(dev, dev["label"], dev["group_id"])):
    splits.loc[dev.index[valid_idx], "fold"] = fold

print(splits["split"].value_counts().to_dict())

## 3. 检查

每一份的类别比例都应接近总体；同一个 `group_id` 只能出现在一份里。

In [ ]:
part = np.where(splits["split"] == "test", "test", "fold " + splits["fold"].astype(str))
classes = ["Negative", "Neutral", "Positive"]
counts = pd.crosstab(part, clean["label_name"])[classes]
counts.loc["all"] = clean["label_name"].value_counts()[classes]
counts["total"] = counts[classes].sum(axis=1)
share = counts[classes].div(counts["total"], axis=0).round(3).add_suffix(" share")
counts.join(share)

In [ ]:
worst = (share.drop(index="all") - share.loc["all"]).abs().max().max()
print(f"各份与总体类别比例的最大差距：{worst:.1%}")
print("跨份的组：", int((splits.groupby("group_id")["fold"].nunique() > 1).sum()))

check_splits(splits, clean)  # 有问题会直接报错
print("check_splits 通过")

## 4. 报告表 5 用的数字

5 折交叉验证时，每一轮用 4 折训练、1 折验证，5 轮轮流；测试集最后只用一次。

In [ ]:
n_test = int((splits["split"] == "test").sum())
n_dev = len(splits) - n_test
fold_sizes = splits.loc[splits["split"] == "train", "fold"].value_counts()


def span(lo, hi, fmt):
    a, b = format(lo, fmt), format(hi, fmt)
    return a if a == b else f"{a}–{b}"


train_lo, train_hi = n_dev - fold_sizes.max(), n_dev - fold_sizes.min()
report_table = pd.DataFrame([
    {"partition": "Training (4 of 5 folds, rotating)", "records": span(train_lo, train_hi, "d"),
     "proportion": span(train_lo / len(splits), train_hi / len(splits), ".1%"), "purpose": "Model and feature fitting"},
    {"partition": "Validation (1 fold, rotating)", "records": span(fold_sizes.min(), fold_sizes.max(), "d"),
     "proportion": span(fold_sizes.min() / len(splits), fold_sizes.max() / len(splits), ".1%"), "purpose": "Model selection (out-of-fold)"},
    {"partition": "Test", "records": str(n_test), "proportion": f"{n_test / len(splits):.1%}", "purpose": "Final evaluation, used once"},
])
report_table

## 5. 保存

划分一旦发布就不再修改。文件已存在且内容不同时会报错，需要新划分时升级 `SPLIT_VERSION`。

In [ ]:
out = SPLITS_DIR / f"folds_v{SPLIT_VERSION}.csv"
splits = splits[["id", "group_id", "split", "fold"]]
if out.exists():
    existing = pd.read_csv(out)
    if not existing.equals(splits):
        raise FileExistsError(f"{out.name} 已存在且内容不同：划分发布后不能修改，请升级 SPLIT_VERSION")
    print(f"{out.name} 已存在，内容一致，未重写")
else:
    splits.to_csv(out, index=False, lineterminator="\n")
    print("已保存", out)

reloaded = load_splits(clean=clean)  # 重新读取，并和清洗数据核对
print(reloaded.shape, reloaded["split"].value_counts().to_dict())